# Company A churn retention proposal

**GCI World 2026 September Final Assignment**

This notebook joins the supplied customer and service records, examines churn, fits a reproducible logistic-regression baseline, and quantifies a clearly labeled pilot scenario. The recommendation is to test assisted retention outreach with a randomized control group; the current model is not strong enough for broad deployment.


## Executive findings

- The one-to-one merge contains **100,000 customers** and 100 columns. The supplied data overview defines churn as exit 31-60 days after the observation date.
- The sample churn share is **49.6%**. This is a cohort rate over the target window; it is not comparable with annual industry churn.
- Churn is **41.7%** when equipment is under 180 days old and **57.9%** at 730 days or older. Churners average 421 equipment-age days versus 363 among non-churners.
- A deterministic 80/20 holdout gives a logistic-regression **ROC-AUC of 0.614**. The highest-risk decile has 67.1% churn, a 1.36x lift, and captures 13.6% of validation churners. This is modest ranking power.
- Recommendation: pilot device-age check-ins and service reviews with a randomized holdout. Do not scale or claim causal impact until recent customer data and incremental retention results are available.


## Business question and target

Can Company A use records available before an observation date to identify customers likely to churn 31-60 days later, then test whether timely support improves retention?

The target is churn in Record.csv. The feature set excludes the target, customer ID, ethnicity, income, credit-card status, marital status, and other demographic fields. These retrospective customer-level findings are not a production decision rule.


## Data setup and reproducibility

Copy Client.csv and Record.csv into data/telecom/, install requirements.txt, and run this notebook from the project root. The analysis checks unique customer IDs and validates the one-to-one join. A fixed integer hash assigns the same 20% holdout each run without looking at the target.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.analysis import run_analysis

ROOT = Path.cwd()
if not (ROOT / "src" / "analysis.py").exists():
    ROOT = ROOT.parent
metrics = run_analysis(ROOT / "data" / "telecom", ROOT / "output")
print(f"Joined rows: {metrics['rows']:,} | columns: {metrics['merged_columns']}")
print(f"Churn rate: {metrics['churn_rate']:.1%}")
print(f"Validation ROC-AUC: {metrics['model']['roc_auc']:.3f}")


## Data completeness

Household and demographic fields are incomplete. The proposed model avoids those fields and uses service, usage, tenure, account, and equipment signals. Continuous values are transformed, clipped, and imputed using training data only; missingness indicators are retained.


In [ ]:
missing = pd.DataFrame(metrics["top_missing"])
missing["share"] = missing["share"].map(lambda x: f"{x:.1%}")
display(missing)


## Equipment age and churn

Observed churn increases across older handset bands. This association supports testing a timely device and plan review; it does not prove that replacing a handset will prevent churn.


In [ ]:
equipment = pd.DataFrame(metrics["churn_by_equipment_age"])
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(equipment["band"], equipment["churn_rate"] * 100, color=["#66B7B1", "#47A9A3", "#268F91", "#125D75"])
ax.set(title="Churn rises with equipment age", xlabel="Equipment age (days)", ylabel="Customers who churned (%)", ylim=(0, 70))
ax.bar_label(bars, labels=[f"{v:.1f}% (n={n:,})" for v, n in zip(equipment["churn_rate"] * 100, equipment["customers"])], padding=4, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()


## Usage patterns

Churners average 483 monthly minutes versus 543 for non-churners (about 11% lower). Average monthly revenue is similar: 58.2 versus 59.2 dataset units. Usage decline is a more promising trigger than revenue level alone in this sample.


In [ ]:
class_summary = metrics["class_summary"]
usage = [class_summary["0"]["mean_monthly_minutes"], class_summary["1"]["mean_monthly_minutes"]]
revenue = [class_summary["0"]["mean_monthly_revenue"], class_summary["1"]["mean_monthly_revenue"]]
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, values, title, ylabel in [(axes[0], usage, "Monthly voice minutes", "Minutes"), (axes[1], revenue, "Mean monthly revenue", "Dataset units")]:
    bars = ax.bar(["Stayed", "Churned"], values, color=["#8BB7C5", "#D89063"])
    ax.bar_label(bars, fmt="%.1f", padding=3, fontsize=9)
    ax.set_title(title); ax.set_ylabel(ylabel); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()


## Model and evaluation

The baseline uses 28 numeric service, usage, account, and equipment features. Preprocessing applies a signed log transform, train-only 1st/99th percentile clipping, median imputation, standardization, and missingness indicators. L2-regularized logistic regression ranks risk without making an automatic contact decision.

ROC-AUC measures ranking across thresholds. The validation split is an ID-hash holdout, not a time-based holdout; production evaluation must use recent future periods.


In [ ]:
model = metrics["model"]
model_summary = pd.DataFrame([
    ("Training rows", f"{model['train_rows']:,}"),
    ("Validation rows", f"{model['validation_rows']:,}"),
    ("Validation churn", f"{model['validation_churn_rate']:.1%}"),
    ("Features including missing flags", str(model["feature_count"])),
    ("ROC-AUC", f"{model['roc_auc']:.3f}"),
    ("Top-decile churn", f"{model['top_decile_churn_rate']:.1%}"),
    ("Top-decile lift", f"{model['top_decile_lift']:.2f}x"),
    ("Churners captured in top decile", f"{model['top_decile_churn_capture']:.1%}"),
], columns=["Measure", "Result"])
display(model_summary)
deciles = pd.DataFrame(model["risk_deciles"])
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(deciles["decile"], deciles["churn_rate"] * 100, color=["#125D75"] + ["#8BB7C5"] * 9)
ax.axhline(model["validation_churn_rate"] * 100, color="#D89063", linestyle="--", linewidth=1.6, label="Validation average")
ax.set(title="Validation churn by predicted-risk decile", xlabel="Risk decile (1 = highest risk)", ylabel="Churn rate (%)", xticks=range(1, 11))
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False); plt.tight_layout()


## Pilot economics: illustrative sensitivity only

For a 100,000-customer population, this scenario contacts the top 10,000, assumes six retained months, uses 55.9 monthly revenue units per top-decile account, and assumes a contact cost of five units. The source data dictionary does not state the currency.

Break-even is a 1.5 percentage-point absolute reduction in 60-day churn before offer discounts, service costs, and discounting. At a 3-point reduction, the scenario implies 300 customers retained and about 100.6k gross revenue units, less 50k contact cost. This is not a forecast: only a randomized experiment can estimate incremental impact.


In [ ]:
scenario = metrics["illustrative_scenario"]
case_table = pd.DataFrame(scenario["cases"])
case_table["absolute_60d_churn_reduction"] = case_table["absolute_60d_churn_reduction"].map(lambda x: f"{x:.0%}")
for col in ["gross_revenue_units_preserved", "contact_cost_units", "net_before_offer_and_service_costs"]:
    case_table[col] = case_table[col].map(lambda x: f"{x:,.0f}")
display(case_table)
print(f"Break-even churn reduction: {scenario['break_even_absolute_churn_reduction']:.1%}")


## Market context and references

The FCC reported 18.5% annual U.S. wireless industry churn for 2023; the course data's 49.6% label rate is a 31-60 day cohort outcome, so these measures are not directly comparable. CTIA reported 132.5 trillion MB of U.S. wireless traffic in 2024, up by more than 32 trillion MB year over year. These sources provide context only; rebuild the model on current, time-stamped Company A data before operational use.

- FCC, 2024 Communications Marketplace Report, FCC 24-136: https://docs.fcc.gov/public/attachments/FCC-24-136A1.pdf
- CTIA, Summary of CTIA's Annual Wireless Industry Survey (2025): https://api.ctia.org/wp-content/uploads/2025/08/2025-CTIA-Survey-Summary-and-Background.pdf
- scikit-learn ROC-AUC documentation: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html


## Decision and safeguards

1. Rebuild the label and predictors from recent time-stamped customer snapshots; check leakage and cohort representativeness.
2. Test device-age check-ins and usage/service reviews with a randomized treatment and control group among eligible higher-risk customers.
3. Measure incremental 60-day retention and net contribution after offers and service costs, not model score alone.
4. Review false positives, contact burden, consent, fairness across geography, and data freshness.
5. Scale only if the experiment demonstrates positive incremental economics.

**AI disclosure:** Codex/ChatGPT assisted with analysis planning, code drafting, and presentation text. The course instructions require a conversation share URL; add this conversation's share URL to the Omnicampus references field before submission.
